<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M08/M08_Lab1_CrewAI_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🤝 M08 Lab 1 — CrewAI Basics</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand the <b>multi-agent</b> paradigm — why multiple specialized agents beat one generalist</li>
    <li>Define <b>agents</b> with roles, goals, and backstories in CrewAI</li>
    <li>Define <b>tasks</b> with descriptions and expected outputs</li>
    <li>Create a <b>crew</b> and run it end-to-end</li>
    <li>Build an <b>investment research crew</b> that produces a professional report</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai crewai crewai-tools
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
import os
from google.colab import userdata
from dads5250 import setup_openai, pp, show_info, compare_responses, quiz

client = setup_openai()
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Why Multi-Agent?</h2>
</div>

In M04-M06, we built **single-agent** systems — one LLM doing everything. But real-world tasks are complex:

| Single Agent | Multi-Agent (CrewAI) |
|-------------|---------------------|
| One prompt does everything | Specialized agents for each subtask |
| Gets confused on complex tasks | Each agent focuses on what it does best |
| No quality control | Agents review each other's work |
| One perspective | Multiple perspectives (researcher + analyst + writer) |

**CrewAI** makes it easy to create teams of AI agents that collaborate. Think of it as building a **virtual team** where each member has a specific role.

### The Three Core Concepts

```
🧑 Agent  = WHO does the work (role, goal, backstory)
📋 Task   = WHAT needs to be done (description, expected output)
👥 Crew   = HOW they work together (process: sequential or hierarchical)
```

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Define Agents</h2>
</div>

Each agent needs:
- **Role**: What is this agent's job title?
- **Goal**: What is this agent trying to achieve?
- **Backstory**: Context that shapes how the agent approaches its work

In [ ]:
# ============================================================
# 🧑 Define Agents with Roles
# ============================================================
from crewai import Agent, Task, Crew, Process

# Agent 1: Researcher
researcher = Agent(
    role="Senior Research Analyst",
    goal="Find comprehensive, accurate information about the given topic",
    backstory="""You are a seasoned research analyst with 15 years of experience 
    in financial markets and technology. You are known for your thorough research 
    methodology and ability to identify key trends and data points that others miss.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# Agent 2: Analyst
analyst = Agent(
    role="Financial Analyst",
    goal="Analyze data and provide actionable investment insights",
    backstory="""You are a CFA charterholder with expertise in fundamental analysis. 
    You excel at interpreting financial data, identifying risks, and making 
    data-driven recommendations. You always back claims with numbers.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# Agent 3: Writer
writer = Agent(
    role="Investment Report Writer",
    goal="Create a clear, professional investment report from research and analysis",
    backstory="""You are an award-winning financial writer who has written for 
    Bloomberg and The Wall Street Journal. You transform complex analysis into 
    clear, compelling reports that executives and investors can act on.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

print("✅ 3 agents created:")
for a in [researcher, analyst, writer]:
    print(f"   🧑 {a.role} — Goal: {a.goal[:60]}...")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> The <b>backstory</b> is more than flavor text — it significantly affects output quality. An agent with "15 years of experience in financial markets" will produce much more detailed, nuanced analysis than one with no backstory. Think of it as the <code>system</code> message on steroids.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Define Tasks</h2>
</div>

Tasks tell agents **what** to do. Each task has:
- **Description**: Detailed instructions
- **Expected output**: What the deliverable should look like
- **Agent**: Who is assigned to this task

In [ ]:
# ============================================================
# 📋 Define Tasks
# ============================================================

STOCK = "NVIDIA"  # Change this to analyze a different company

# Task 1: Research
research_task = Task(
    description=f"""Research {STOCK} comprehensively. Include:
    1. Current stock price and recent performance (last 6 months)
    2. Key financial metrics (revenue, earnings, P/E ratio, market cap)
    3. Recent news and developments (last 3 months)
    4. Competitive landscape and market position
    5. Major risks and opportunities""",
    expected_output=f"A detailed research brief on {STOCK} covering financials, news, competition, and risk factors.",
    agent=researcher
)

# Task 2: Analyze
analysis_task = Task(
    description=f"""Using the research provided, perform a thorough financial analysis of {STOCK}:
    1. Evaluate the company's financial health (profitability, growth, debt)
    2. Assess valuation (is it overvalued/undervalued?)
    3. Identify 3 bull case factors and 3 bear case factors
    4. Provide a price target range (12-month)
    5. Rate overall investment attractiveness on a 1-10 scale with justification""",
    expected_output=f"A structured financial analysis with bull/bear cases, price target, and investment rating.",
    agent=analyst
)

# Task 3: Write Report
report_task = Task(
    description=f"""Create a professional investment report for {STOCK} based on the research and analysis.
    
    Format the report with these sections:
    - Executive Summary (2-3 sentences)
    - Company Overview
    - Financial Analysis
    - Bull Case / Bear Case
    - Investment Recommendation (BUY / HOLD / SELL) with price target
    - Risk Disclosure
    
    Keep it concise but comprehensive. Use bullet points for readability.""",
    expected_output="A complete, professionally formatted investment report ready for presentation.",
    agent=writer
)

print(f"✅ 3 tasks defined for {STOCK} analysis")
print(f"   📋 Research → Analyze → Write Report")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Create & Run the Crew</h2>
</div>

Now we assemble the crew and let them work! The **sequential** process means each agent works in order, passing results to the next.

In [ ]:
# ============================================================
# 👥 Create and Run the Crew
# ============================================================

investment_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.sequential,  # Run tasks in order
    verbose=True
)

print(f"🚀 Running investment analysis crew for {STOCK}...")
print(f"   This will make ~3 LLM calls (one per agent).\n")

result = investment_crew.kickoff()

print("\n" + "="*70)
print("✅ CREW COMPLETE — Final Report:")
print("="*70)

In [ ]:
# ============================================================
# 📊 Display the Investment Report
# ============================================================
from IPython.display import display, HTML, Markdown

# Display as formatted markdown
display(Markdown(str(result)))

**📝 Your Observations:** *(double-click to edit)*

1. Read the output. Can you tell which agent contributed which part of the report? _[Your answer]_
2. Did the writer's report incorporate specific data from the researcher and analyst? _[Your answer]_
3. What would happen if you reversed the agent order (writer first, researcher last)? _[Your answer]_

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Notice how the <b>writer agent</b> produced a much more polished report than if you had asked a single LLM to do everything. That's the power of specialization: the researcher gathers facts, the analyst interprets them, and the writer presents them clearly.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "In CrewAI, what is the purpose of an agent's 'backstory'?",
        "options": [
            "It's optional decoration that doesn't affect output",
            "It provides context and expertise that shapes how the agent approaches tasks — like a detailed system prompt",
            "It stores the agent's conversation history",
            "It defines what tools the agent can use"
        ],
        "answer": 1,
        "explanation": "The backstory acts as a rich system prompt that defines the agent's expertise, experience, and approach. A 'CFA charterholder with 10 years experience' will produce very different analysis than a generic agent."
    },
    {
        "q": "What is the difference between 'sequential' and 'hierarchical' crew processes?",
        "options": [
            "Sequential is faster, hierarchical is slower",
            "Sequential runs tasks in order (each agent passes output to the next); hierarchical has a manager agent that delegates",
            "Sequential uses one LLM, hierarchical uses multiple LLMs",
            "There is no difference — they produce the same output"
        ],
        "answer": 1,
        "explanation": "Sequential process runs tasks in a fixed order (agent 1 → 2 → 3). Hierarchical process adds a manager agent that dynamically delegates tasks and reviews quality — useful for more complex workflows."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add a Risk Assessor Agent

Add a fourth agent to the crew — a **Risk Assessor** who evaluates the investment risks before the writer creates the report.

In [ ]:
# --- Expected Output ---
show_expected("A 4-agent crew (Researcher + Analyst + Risk Assessor + Writer) producing a comprehensive investment report with risk assessment section.")

In [ ]:
# ============================================================
# 🔧 Exercise: Add a Risk Assessor Agent
# ============================================================
# Replace each ----- with the correct value

# New agent: Risk Assessor
risk_assessor = Agent(
    role="-----",                  # Give it a role title
    goal="-----",                  # What is this agent trying to achieve?
    backstory="""You are a risk management expert with experience at major 
    investment banks. You specialize in identifying hidden risks that others 
    overlook — regulatory, geopolitical, technological, and market risks.""",
    verbose=True,
    allow_delegation=False,
    llm="gpt-4.1-mini"
)

# New task: Risk Assessment
risk_task = Task(
    description=f"""Assess the investment risks of {STOCK}:
    1. Market risks (competition, market saturation)
    2. Financial risks (debt, cash flow, valuation)
    3. Regulatory risks
    4. Technology risks (disruption, obsolescence)
    5. Rate overall risk as LOW / MEDIUM / HIGH with justification""",
    expected_output="A structured risk assessment with risk categories and overall rating.",
    agent=-----                     # Which agent handles this?
)

# Updated crew with 4 agents
expanded_crew = Crew(
    agents=[researcher, analyst, risk_assessor, writer],
    tasks=[research_task, analysis_task, risk_task, report_task],
    process=Process.-----,          # What process type?
    verbose=True
)

print("✅ Expanded crew created with 4 agents!")
print("   Run expanded_crew.kickoff() to execute (takes ~30 seconds)")

# Uncomment to run:
# expanded_result = expanded_crew.kickoff()
# display(Markdown(str(expanded_result)))

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the risk assessor add value that was missing from the original 3-agent crew? _[Your answer]_

2. How did the writer incorporate the risk assessment into the final report? _[Your answer]_

3. Did you notice any contradictions between the analyst (bullish) and risk assessor (cautious)? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions:</p>
  <ol style="font-size: 14px;">
    <li>Change <code>STOCK</code> to a different company and re-run — does the quality hold up?</li>
    <li>Try <code>Process.hierarchical</code> instead of sequential — how does it differ?</li>
    <li>Add a <b>Fact Checker</b> agent that verifies the report before finalizing</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've built a multi-agent investment research crew with CrewAI.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Agents</b> have roles, goals, and backstories that shape their behavior</li>
      <li><b>Tasks</b> define what each agent does and what output to expect</li>
      <li><b>Crews</b> orchestrate agents — sequential or hierarchical processes</li>
      <li>Multi-agent systems produce higher quality output through <b>specialization</b></li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M08 Lab 2: Multi-Agent Content Pipeline</p>
</div>